# Глава I. Адаптивная устойчивость российского общества — вычислительный эксперимент FIX4.1 FINAL

**Назначение:** финальный воспроизводимый эксперимент, подготовленный к научной приемке и оформлению результатов исследования.

FIX4.1 сохраняет устойчивую архитектуру FIX4 (LIVE → структурный QC → встроенный официальный snapshot), но исправляет статистический слой: малые выборки проверяются exact/permutation-тестами; внешняя валидизация проводится стратифицированно внутри типов социальных групп; change-point detection реализован как точное penalized dynamic programming (L2) без некорректного обозначения PELT; все множественные проверки проходят Benjamini–Hochberg FDR.

## 1. Исследовательская модель

Основная единица агрегированного анализа — **волна × социальная группа × показатель**. Respondent-level ML/SHAP запускается только при обнаружении настоящих микроданных.

Баланс ожиданий:

$$B_{g,t}=p_{\mathrm{better},g,t}-p_{\mathrm{worse},g,t}.$$

Микро–макро разрыв внутри одной полевой волны:

$$G^{MM}_t=B_{\mathrm{family},t}-B_{\mathrm{country},t}.$$

Контраст горизонтов сравнивает **соседние независимые волны**, а не ответы одних и тех же респондентов:

$$G^{H}_t=B_{\mathrm{long},t+\Delta}-B_{\mathrm{short},t}.$$

Для малых $n$ статистическая значимость корреляций определяется перестановочно. При полном переборе:

$$p_{\mathrm{exact}}=\frac{\#\{|T_\pi|\ge |T_{obs}|\}}{N_\pi}.$$

При Monte-Carlo permutation:

$$p_{MC}=\frac{1+\#\{|T_b|\ge |T_{obs}|\}}{B+1}.$$

Внешняя валидизация не объединяет возраст, образование, округа и другие системы разбиения как независимые наблюдения. Ранги центрируются и стандартизируются **внутри `group_type`**, после чего перестановки выполняются только внутри соответствующего блока. Для множественных проверок применяется Benjamini–Hochberg FDR.

Change-point detection минимизирует:

$$\min_{\tau_1,\ldots,\tau_m}\left[\sum_{j=0}^{m} SSE(\tau_j,\tau_{j+1})+\beta m\right],$$

где оптимум находится точным динамическим программированием (`EXACT_DP_L2`). Это корректное название фактически используемого метода; PELT в FIX4.1 не заявляется.

In [ ]:
# 0. ИМПОРТЫ И КОНФИГУРАЦИЯ — FIX4.1 FINAL
import os,re,io,json,math,time,shutil,zipfile,hashlib,base64,warnings,importlib.util,itertools
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor,as_completed
warnings.filterwarnings("ignore")
import numpy as np,pandas as pd,requests,matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from scipy.stats import spearmanr,theilslopes,rankdata
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score,adjusted_rand_score,f1_score,balanced_accuracy_score
try: import ruptures as rpt; HAS_RUPTURES=True
except Exception: HAS_RUPTURES=False
try: import umap; HAS_UMAP=True
except Exception: HAS_UMAP=False
try: import pyreadstat; HAS_PYREADSTAT=True
except Exception: HAS_PYREADSTAT=False
try: from lightgbm import LGBMClassifier; HAS_LGBM=True
except Exception: HAS_LGBM=False
try: from catboost import CatBoostClassifier; HAS_CATBOOST=True
except Exception: HAS_CATBOOST=False
try: import shap; HAS_SHAP=True
except Exception: HAS_SHAP=False

SEED=42; np.random.seed(SEED)
ROOT=Path('/content/MONOGRAPH_CH1_FIX41'); LIVE=ROOT/'raw_live'; SNAP=ROOT/'snapshot'; RESULTS=ROOT/'results'; FIG=RESULTS/'figures'; TABLES=RESULTS/'tables'
for p in [LIVE,SNAP,RESULTS,FIG,TABLES]: p.mkdir(parents=True,exist_ok=True)
CONFIG={"try_live":True,"live_timeout":18,"year_min":2017,"bootstrap_n":800,"cluster_bootstrap_n":500,"cluster_seed_n":30,"min_trend_n":7,"min_changepoint_n":8,"min_cluster_coverage":0.80,"min_cluster_groups":12,"response_sum_tolerance":1.0,"html_binary_balance_tolerance":1.25,"validation_expectation_source":15317,"run_ml":True,"run_shap":True,"min_micro_n":500,"min_class_n":30,"test_size":0.25,"perm_n":20000,"cross_perm_n":10000,"exact_perm_max_n":8,"validation_min_block_n":3,"validation_min_blocks":4,"validation_sign_consistency":0.70}
SOURCES={15135:{"kind":"anxiety_history","page":"https://fom.ru/Nastroeniya/15135","snapshot":"15135.html","publication":"2025-02-04"},15297:{"kind":"anxiety","page":"https://fom.ru/Nastroeniya/15297","download":"https://fom.ru/posts/download/15297","snapshot":"15297.xlsx","publication":"2026-01-30"},15305:{"kind":"validation","page":"https://fom.ru/Mir/15305","download":"https://fom.ru/posts/download/15305","snapshot":"15305.xlsx","publication":"2026-02-17"},15277:{"kind":"expectations","page":"https://fom.ru/Nastroeniya/15277","download":"https://fom.ru/posts/download/15277","snapshot":"15277.xlsx","publication":"2025-12-11"},15317:{"kind":"expectations","page":"https://fom.ru/Nastroeniya/15317","download":"https://fom.ru/posts/download/15317","snapshot":"15317.xlsx","publication":"2026-03-26"},15337:{"kind":"expectations","page":"https://fom.ru/Nastroeniya/15337","download":"https://fom.ru/posts/download/15337","snapshot":"15337.xlsx","publication":"2026-05-12"},15357:{"kind":"expectations","page":"https://fom.ru/Nastroeniya/15357","download":"https://fom.ru/posts/download/15357","snapshot":"15357.xlsx","publication":"2026-07-14"},15380:{"kind":"expectations","page":"https://fom.ru/Nastroeniya/15380","download":"https://fom.ru/posts/download/15380","snapshot":"15380.xlsx","history_snapshot":"15380.html","publication":"2026-09-15"}}
UA='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/154 Safari/537.36'; SESSION=requests.Session(); SESSION.headers.update({'User-Agent':UA,'Accept-Language':'ru-RU,ru;q=0.9,en;q=0.5'})


In [ ]:
# 1. ВСТРОЕННЫЙ ОФИЦИАЛЬНЫЙ SNAPSHOT — FIX4.1, НЕ РЕДАКТИРОВАТЬ
SNAPSHOT_ZIP_SHA256='9d294fbb399e96f74b41d79dd0d0cc90c63697068ebc4ec23921bc4b3cc56598'; SNAPSHOT_FILE_SHA256={'15277.xlsx': 'e601ccfbfa8b79f3d002e0cf6fc4087cddfbc5d61e4a85dfebd0cab4b4a0c0c3', '15317.xlsx': 'c975c661e068d00dd4a7a6526282a15a5c83bfe37db2d3268a8b270ed6aa14bd', '15337.xlsx': 'f0b0df862a1a7fc9a2e4ccc176bc7e1182eff8721c4da07f0677b12c6b8724a5', '15357.xlsx': 'f8e80d25f5f69d258727be5476f4fd710537094308bf7fd7027e82023aba8280', '15380.xlsx': 'abd2db18f2ac8c408f2ebfa666ac7e34a1625d9cef41789d88e8998883367ba0', '15297.xlsx': '0bf4f303becaa00c71f7086008de5ddcd7aff288b5f8a2ef24478292abbcc2eb', '15305.xlsx': '94c98f2e6fd6fb45fc5375c55401af7d65234c7e276381b9ef5e817353a68cc5', '15380.html': '28bedea50154a34cd11f649c7fddfbbc66aff9263b3275afdd5c03854805ee3e', '15135.html': 'aef37bb6d0c47f23428b87624a47d9e66362415318dba6cbb37623b6c00ea4c7'}; SNAPSHOT_ORIGIN_META={"15305_original_xls_sha256":'cea2a8ca5e01499223779ea7cd2b89526f3fd913a020238e7b06e73d93b9c4b7',"15305_snapshot_note":"официальный 15305.xls технически конвертирован в XLSX для независимости от xlrd"}
SNAPSHOT_ZIP_B64='...'
def materialize_snapshot():
    b=base64.b64decode(SNAPSHOT_ZIP_B64); assert hashlib.sha256(b).hexdigest()==SNAPSHOT_ZIP_SHA256,"Повреждён встроенный FIX4 snapshot"; shutil.rmtree(SNAP,ignore_errors=True); SNAP.mkdir(parents=True,exist_ok=True); zipfile.ZipFile(io.BytesIO(b)).extractall(SNAP); bad=[]
    for name,h in SNAPSHOT_FILE_SHA256.items():
        p=SNAP/name; bad.append(name) if (not p.exists() or hashlib.sha256(p.read_bytes()).hexdigest()!=h) else None
    if bad: raise RuntimeError('Ошибка хэшей встроенного snapshot: '+', '.join(bad))
    return pd.DataFrame([{"file":k,"sha256":v,"bytes":(SNAP/k).stat().st_size} for k,v in SNAPSHOT_FILE_SHA256.items()])


## 2. Слой данных и приемочный шлюз

Каждый обязательный блок проходит одинаковую последовательность: **LIVE → структурный parser/QC → EMBEDDED_OFFICIAL_SNAPSHOT при недоступности или структурной непригодности live-источника**. Режим источника, SHA-256 и результат QC фиксируются в аудитах.

Snapshot является резервом воспроизводимости, а не скрытой заменой данных. Итоговый отчет отдельно показывает, использовался ли он в конкретном запуске.

In [ ]:
# 2. ИСТОЧНИКИ, ДАТЫ, ГРУППЫ, УНИВЕРСАЛЬНОЕ ЧТЕНИЕ
MONTHS={"января":1,"февраля":2,"марта":3,"апреля":4,"мая":5,"июня":6,"июля":7,"августа":8,"сентября":9,"октября":10,"ноября":11,"декабря":12}
def clean_text(x):
    if x is None or (isinstance(x,float) and np.isnan(x)): return ""
    return re.sub(r"\s+"," ",str(x).replace("\xa0"," ").replace("ё","е").replace("–","-").replace("—","-")).strip()
def norm_token(x): return re.sub(r"\s+"," ",re.sub(r"[^0-9a-zа-я]+"," ",clean_text(x).lower())).strip()
def sha256_file(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def publication_date_for(sid): return pd.to_datetime(SOURCES[int(sid)]["publication"])
def sniff_ext(b,ct=""):
    if b[:4]==b"PK\x03\x04":
        try:
            n=set(zipfile.ZipFile(io.BytesIO(b)).namelist()); return ".xlsx" if "[Content_Types].xml" in n and any(x.startswith("xl/") for x in n) else ".zip"
        except Exception: return ".zip"
    if b[:8]==bytes.fromhex("D0CF11E0A1B11AE1"): return ".xls"
    if b[:4] in (b"$FL2",b"$FL3"): return ".sav"
    h=b[:500].lower(); return ".html" if b"<html" in h or b"<!doctype" in h or "text/html" in (ct or "").lower() else ".bin"
def parse_fieldwork_range(text):
    s=clean_text(text).lower(); m=re.search(r"(\d{1,2})\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s*-\s*(\d{1,2})\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+(\d{4})",s)
    if m: d1,mo1,d2,mo2,y=int(m.group(1)),MONTHS[m.group(2)],int(m.group(3)),MONTHS[m.group(4)],int(m.group(5)); a=pd.Timestamp(y if mo1<=mo2 else y-1,mo1,d1); b=pd.Timestamp(y,mo2,d2); return a,b,a+(b-a)/2
    m=re.search(r"(\d{1,2})\s*-\s*(\d{1,2})\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+(\d{4})",s)
    if m: a=pd.Timestamp(int(m.group(4)),MONTHS[m.group(3)],int(m.group(1))); b=pd.Timestamp(int(m.group(4)),MONTHS[m.group(3)],int(m.group(2))); return a,b,a+(b-a)/2
    m=re.search(r"(\d{1,2})\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+(\d{4})",s)
    if m: a=pd.Timestamp(int(m.group(3)),MONTHS[m.group(2)],int(m.group(1))); return a,a,a
    return pd.NaT,pd.NaT,pd.NaT
def parse_ru_date_label(label):
    s=clean_text(label).lower()
    if re.fullmatch(r"\d{4}",s): y=int(s); return pd.Timestamp(y,7,1),"annual_average",y
    m=re.search(r"(\d{1,2})\s+(января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\s+(\d{4})",s)
    if m: y=int(m.group(3)); return pd.Timestamp(y,MONTHS[m.group(2)],int(m.group(1))),"wave",y
    return pd.NaT,"unknown",np.nan
def group_type_from_header(h1,h2=""):
    a=clean_text(h1).lower(); b=clean_text(h2).lower()
    if "все опрошенные" in a or ("население" in a and "целом" in a): return "overall"
    if "федеральн" in a: return "federal_district"
    if "возраст" in a and "образован" in a: return "age_education"
    if "возраст" in a: return "age"
    if "образован" in a: return "education"
    if "материаль" in a or "имуществен" in a: return "material_status"
    if "тип населен" in a: return "settlement"
    if "источник" in a and "информац" in a: return "media_source"
    if a.strip()=="пол": return "gender"
    if "занятост" in a: return "employment"
    if "доход" in a: return "income"
    return "other"
def normalize_group_key(gt,name):
    s=norm_token(name); aliases={"из интернета":"интернет","из традиционных сми":"традиционные сми"}; s=aliases.get(s,s); return s
def header_for_question(df,qrow):
    share=next((r for r in range(qrow-1,-1,-1) if clean_text(df.iloc[r,0]).lower()=="доли групп"),None)
    if share is None or share<2: return []
    h1=[clean_text(x) for x in df.iloc[share-2,:].tolist()]; h2=[clean_text(x) for x in df.iloc[share-1,:].tolist()]; ff=[]; cur=""
    for x in h1: cur=x or cur; ff.append(cur)
    out=[]
    for c in range(1,df.shape[1]):
        top=ff[c] if c<len(ff) else ""; sub=h2[c] if c<len(h2) else ""; name=sub or top
        if name: gt=group_type_from_header(top,sub); out.append((c,gt,clean_text(name),normalize_group_key(gt,name),top))
    return out
def read_workbook(path):
    p=Path(path); out={}
    if p.suffix.lower() not in (".xlsx",".xls"): return out
    try: xls=pd.ExcelFile(p)
    except Exception: return out
    for sh in xls.sheet_names:
        try: out[str(sh)]=pd.read_excel(p,sheet_name=sh,header=None,dtype=object)
        except Exception: pass
    return out
def request_once(url,timeout=None):
    r=SESSION.get(url,timeout=timeout or CONFIG["live_timeout"],allow_redirects=True); r.raise_for_status(); return r
def live_fetch_one(sid):
    src=SOURCES[sid]; folder=LIVE/str(sid); folder.mkdir(parents=True,exist_ok=True); rec={"source_id":sid,"live_download":"","live_page":"","live_error":""}
    if not CONFIG["try_live"]: rec["live_error"]="LIVE_DISABLED"; return rec
    if src.get("download"):
        try:
            r=request_once(src["download"]); b=r.content; ext=sniff_ext(b,r.headers.get("content-type","")); p=folder/f"download{ext}"; p.write_bytes(b); rec["live_download"]=str(p)
            if ext==".zip":
                try: zipfile.ZipFile(p).extractall(folder/"unzipped")
                except Exception as e: rec["live_error"]+=f"ZIP:{type(e).__name__};"
        except Exception as e: rec["live_error"]+=f"DOWNLOAD:{type(e).__name__};"
    if sid in (15135,15380):
        try:
            r=request_once(src["page"]); txt=r.text
            if len(txt)>1200: p=folder/"page.html"; p.write_text(txt,encoding="utf-8"); rec["live_page"]=str(p)
        except Exception as e: rec["live_error"]+=f"PAGE:{type(e).__name__};"
    return rec
def fetch_live_parallel():
    ids=sorted(SOURCES); rows=[]
    with ThreadPoolExecutor(max_workers=min(8,len(ids))) as ex:
        fut={ex.submit(live_fetch_one,s):s for s in ids}
        for f in as_completed(fut):
            try: rows.append(f.result())
            except Exception as e: rows.append({"source_id":fut[f],"live_download":"","live_page":"","live_error":repr(e)[:160]})
    return pd.DataFrame(rows).sort_values("source_id").reset_index(drop=True)
def workbook_candidates(sid,live_audit):
    out=[]; q=live_audit[live_audit.source_id.eq(sid)] if len(live_audit) else pd.DataFrame()
    if len(q) and q.iloc[0].live_download:
        p=Path(q.iloc[0].live_download)
        if p.suffix.lower() in (".xlsx",".xls"): out.append(("LIVE",p))
        elif p.suffix.lower()==".zip": out += [("LIVE",x) for x in p.parent.joinpath("unzipped").rglob("*") if x.is_file() and x.suffix.lower() in (".xlsx",".xls")]
    out.append(("EMBEDDED_OFFICIAL_SNAPSHOT",SNAP/SOURCES[sid]["snapshot"])); return out
def html_candidates(sid,live_audit,history=False):
    out=[]; q=live_audit[live_audit.source_id.eq(sid)] if len(live_audit) else pd.DataFrame()
    if len(q) and q.iloc[0].live_page: out.append(("LIVE",Path(q.iloc[0].live_page)))
    name=SOURCES[sid].get("history_snapshot") if history else SOURCES[sid].get("snapshot"); out.append(("EMBEDDED_OFFICIAL_SNAPSHOT",SNAP/name)); return out


In [ ]:
# 3. СТРУКТУРНЫЕ ПАРСЕРЫ И PARSE-AWARE FALLBACK
def construct_from_question(q):
    s=clean_text(q).lower(); subject="family" if ("ваша жизнь" in s or "жизнь вашей семьи" in s) else ("country" if "нашей стране" in s else None); horizon="short" if re.search(r"полгода\s*-?\s*год",s) else ("long" if re.search(r"(три\s*-?\s*пять|3\s*-?\s*5)\s+лет",s) else None); return subject,horizon
def expectation_answer_class(a):
    s=clean_text(a).lower(); return "better" if "станет лучше" in s else ("worse" if "станет хуже" in s else ("same" if "не изменится" in s else ("dk" if "затрудня" in s else None)))
def parse_expectation_workbook(df,sid):
    rows=[]; pub=publication_date_for(sid)
    for r in range(len(df)):
        q=clean_text(df.iloc[r,0]); subject,horizon=construct_from_question(q)
        if not (subject and horizon): continue
        found={}
        for rr in range(r+1,min(r+8,len(df))):
            cls=expectation_answer_class(df.iloc[rr,0]); found.setdefault(cls,rr) if cls else None
        if set(found)!={"better","worse","same","dk"}: continue
        fw=(pd.NaT,pd.NaT,pd.NaT)
        for rr in range(r-1,max(-1,r-55),-1):
            txt=clean_text(df.iloc[rr,0])
            if "фомнибус" in txt.lower() or "опрос" in txt.lower():
                z=parse_fieldwork_range(txt)
                if pd.notna(z[0]): fw=z; break
        for c,gt,gname,gkey,_ in header_for_question(df,r):
            vals={k:pd.to_numeric(pd.Series([df.iloc[rr,c]]),errors="coerce").iloc[0] for k,rr in found.items()}
            if all(pd.isna(v) for v in vals.values()): continue
            vals={k:(float(v) if pd.notna(v) else np.nan) for k,v in vals.items()}; rows.append({"source_id":sid,"publication_date":pub,"fieldwork_start":fw[0],"fieldwork_end":fw[1],"wave_date":fw[2],"question":q,"subject":subject,"horizon":horizon,"construct":f"{subject}_{horizon}","group_type":gt,"group":gname,"group_key":gkey,**vals,"response_sum":sum(v for v in vals.values() if pd.notna(v)),"balance":vals["better"]-vals["worse"] if pd.notna(vals["better"]) and pd.notna(vals["worse"]) else np.nan})
    return pd.DataFrame(rows)
def valid_expectation_source(x):
    if x is None or x.empty: return False
    req={"country_short","family_short","country_long","family_long"}; ov=x[x.group_type.eq("overall")]; return set(x.construct.dropna())==req and len(ov[["construct"]].drop_duplicates())==4 and (x.response_sum-100).abs().max()<=CONFIG["response_sum_tolerance"]
def choose_expectation_source(sid,live_audit):
    attempts=[]
    for mode,p in workbook_candidates(sid,live_audit):
        try:
            frames=[parse_expectation_workbook(df,sid) for df in read_workbook(p).values()]; x=pd.concat([z for z in frames if len(z)],ignore_index=True) if any(len(z) for z in frames) else pd.DataFrame(); ok=valid_expectation_source(x); attempts.append({"source_id":sid,"block":"expectations","mode":mode,"path":str(p),"rows":len(x),"status":"PASS" if ok else "REJECTED","sha256":sha256_file(p) if p.exists() else ""})
            if ok: return x,pd.DataFrame(attempts)
        except Exception as e: attempts.append({"source_id":sid,"block":"expectations","mode":mode,"path":str(p),"rows":0,"status":"ERROR","detail":repr(e)[:180]})
    return pd.DataFrame(),pd.DataFrame(attempts)
def parse_expectation_hcparams(html,sid=15380):
    soup=BeautifulSoup(html,"lxml"); out=[]
    for h2 in soup.find_all("h2",class_="chart-header"):
        q=clean_text(h2.get_text(" ",strip=True)); subject,horizon=construct_from_question(q)
        if not (subject and horizon): continue
        payload=None; node=h2
        while True:
            node=node.find_next()
            if node is None or (node.name=="h2" and "chart-header" in (node.get("class") or [])): break
            if node.name=="script":
                m=re.search(r"var\s+HCparams\s*=\s*(\{.*?\});",node.string or node.get_text(),re.S)
                if m:
                    try: payload=json.loads(m.group(1)); break
                    except Exception: pass
        if not payload: continue
        cats=payload.get("categoies") or payload.get("categories") or []; series=payload.get("series") or []; values=payload.get("values") or []
        for si,label in enumerate(series):
            cls=expectation_answer_class(label)
            if not cls or si>=len(values): continue
            for j,cat in enumerate(cats):
                if j>=len(values[si]): continue
                dt,ptype,year=parse_ru_date_label(cat)
                if pd.isna(year) or int(year)<CONFIG["year_min"]: continue
                out.append({"source_id":sid,"question":q,"subject":subject,"horizon":horizon,"construct":f"{subject}_{horizon}","time_label":str(cat),"observation_date":dt,"year":int(year),"period_type":ptype,"answer":cls,"value":float(values[si][j])})
    long=pd.DataFrame(out)
    if long.empty: return long,pd.DataFrame(),pd.DataFrame()
    idx=["source_id","question","subject","horizon","construct","time_label","observation_date","year","period_type"]; points=long.pivot_table(index=idx,columns="answer",values="value",aggfunc="first").reset_index()
    for c in ["better","worse","same","dk"]:
        if c not in points: points[c]=np.nan
    points["balance"]=points.better-points.worse; annual0=points[points.period_type.eq("annual_average")].copy(); annual0["waves_n"]=np.nan; wave=points[points.period_type.eq("wave")]; annualw=wave.groupby(["subject","horizon","construct","year"],as_index=False).agg(better=("better","mean"),worse=("worse","mean"),same=("same","mean"),dk=("dk","mean"),balance=("balance","mean"),waves_n=("balance","size")); annual=pd.concat([annual0[["subject","horizon","construct","year","better","worse","same","dk","balance","waves_n"]],annualw],ignore_index=True); annual["priority"]=annual.waves_n.notna().astype(int); annual=annual.sort_values(["construct","year","priority"]).drop_duplicates(["construct","year"],keep="first").drop(columns="priority").sort_values(["construct","year"]).reset_index(drop=True); return long,points,annual
def valid_expectation_history(points,annual):
    req={"country_short","family_short","country_long","family_long"}
    if points is None or annual is None or points.empty or annual.empty or not req.issubset(set(annual.construct)): return False
    cov=annual.groupby("construct").year.nunique().to_dict(); return all(cov.get(c,0)>=10 for c in req)
def choose_expectation_history(live_audit):
    attempts=[]
    for mode,p in html_candidates(15380,live_audit,history=True):
        try:
            l,pt,a=parse_expectation_hcparams(p.read_text(encoding="utf-8",errors="ignore"),15380); ok=valid_expectation_history(pt,a); attempts.append({"source_id":15380,"block":"expectation_history","mode":mode,"path":str(p),"rows":len(a),"status":"PASS" if ok else "REJECTED","sha256":sha256_file(p)})
            if ok: return l,pt,a,pd.DataFrame(attempts)
        except Exception as e: attempts.append({"source_id":15380,"block":"expectation_history","mode":mode,"path":str(p),"rows":0,"status":"ERROR","detail":repr(e)[:180]})
    return pd.DataFrame(),pd.DataFrame(),pd.DataFrame(),pd.DataFrame(attempts)
def anxiety_subject_from_question(q):
    s=clean_text(q).lower(); return "personal" if "повседневной жизни" in s else ("country" if "ситуации в россии" in s else ("world" if "ситуации в мире" in s else None))
def parse_anxiety_workbook(df,sid=15297):
    rows=[]; pub=publication_date_for(sid); qrows=[(r,anxiety_subject_from_question(df.iloc[r,0]),clean_text(df.iloc[r,0])) for r in range(len(df)) if anxiety_subject_from_question(df.iloc[r,0])]
    for qi,(r,subject,q) in enumerate(qrows):
        stop=qrows[qi+1][0] if qi+1<len(qrows) else len(df); fw=(pd.NaT,pd.NaT,pd.NaT)
        for rr in range(r-1,max(-1,r-55),-1):
            txt=clean_text(df.iloc[rr,0])
            if "фомнибус" in txt.lower():
                z=parse_fieldwork_range(txt)
                if pd.notna(z[0]): fw=z; break
        groups=header_for_question(df,r)
        for rr in range(r+1,stop):
            item=clean_text(df.iloc[rr,0]); nums=pd.to_numeric(df.iloc[rr,1:],errors="coerce")
            if not item or item.lower()=="доли групп" or nums.notna().sum()==0: continue
            for c,gt,gname,gkey,_ in groups:
                v=pd.to_numeric(pd.Series([df.iloc[rr,c]]),errors="coerce").iloc[0]
                if pd.notna(v): rows.append({"source_id":sid,"publication_date":pub,"fieldwork_start":fw[0],"fieldwork_end":fw[1],"wave_date":fw[2],"subject":subject,"question":q,"anxiety_item":item,"item_key":norm_token(item),"group_type":gt,"group":gname,"group_key":gkey,"value":float(v)})
    return pd.DataFrame(rows)
def valid_anxiety_current(x):
    if x is None or x.empty: return False
    ov=x[x.group_type.eq("overall")]; cnt=ov.groupby("subject").size().to_dict(); return set(ov.subject)=={"personal","country","world"} and all(cnt.get(s,0)>=10 for s in ["personal","country","world"]) and not ov.anxiety_item.str.lower().eq("доли групп").any()
def choose_anxiety_current(live_audit):
    attempts=[]
    for mode,p in workbook_candidates(15297,live_audit):
        try:
            frames=[parse_anxiety_workbook(df,15297) for df in read_workbook(p).values()]; x=pd.concat([z for z in frames if len(z)],ignore_index=True) if any(len(z) for z in frames) else pd.DataFrame(); ok=valid_anxiety_current(x); attempts.append({"source_id":15297,"block":"anxiety_current","mode":mode,"path":str(p),"rows":len(x),"status":"PASS" if ok else "REJECTED","sha256":sha256_file(p)})
            if ok: return x,pd.DataFrame(attempts)
        except Exception as e: attempts.append({"source_id":15297,"block":"anxiety_current","mode":mode,"path":str(p),"rows":0,"status":"ERROR","detail":repr(e)[:180]})
    return pd.DataFrame(),pd.DataFrame(attempts)
def parse_anxiety_history_html(html,sid=15135):
    out=[]; signatures={"personal":["здоровье близких","собственное здоровье"],"country":["угроза нападения других государств","повышение цен"],"world":["ядерная война","международный терроризм"]}
    try: tabs=pd.read_html(io.StringIO(html))
    except Exception: tabs=[]
    for df in tabs:
        if df.shape[1]<3: continue
        col0=[clean_text(x).lower() for x in df.iloc[:,0].tolist()]; subject=next((s for s,keys in signatures.items() if any(any(k in x for x in col0[:14]) for k in keys)),None)
        if subject is None: continue
        dates=[]
        for c in range(1,df.shape[1]):
            lab=clean_text(df.iloc[0,c]); dt,_,year=parse_ru_date_label(lab)
            if pd.notna(dt): dates.append((c,lab,dt,int(year)))
        for r in range(1,len(df)):
            item=clean_text(df.iloc[r,0])
            if not item or item.lower()=="доли групп": continue
            for c,lab,dt,year in dates:
                v=pd.to_numeric(pd.Series([df.iloc[r,c]]),errors="coerce").iloc[0]
                if pd.notna(v): out.append({"source_id":sid,"subject":subject,"anxiety_item":item,"item_key":norm_token(item),"observation_date":dt,"year":year,"time_label":lab,"value":float(v)})
    return pd.DataFrame(out)
def valid_anxiety_history(x):
    if x is None or x.empty: return False
    cov=x.groupby("subject").year.nunique().to_dict(); return set(x.subject)=={"personal","country","world"} and all(cov.get(s,0)>=10 for s in ["personal","country","world"])
def choose_anxiety_history(live_audit):
    attempts=[]
    for mode,p in html_candidates(15135,live_audit):
        try:
            x=parse_anxiety_history_html(p.read_text(encoding="utf-8",errors="ignore"),15135); ok=valid_anxiety_history(x); attempts.append({"source_id":15135,"block":"anxiety_history","mode":mode,"path":str(p),"rows":len(x),"status":"PASS" if ok else "REJECTED","sha256":sha256_file(p)})
            if ok: return x,pd.DataFrame(attempts)
        except Exception as e: attempts.append({"source_id":15135,"block":"anxiety_history","mode":mode,"path":str(p),"rows":0,"status":"ERROR","detail":repr(e)[:180]})
    return pd.DataFrame(),pd.DataFrame(attempts)
VALIDATION_SPECS=[("developed_country",r"мы живем скорее в развитой",["развитой, передовой"],["неразвитой, отсталой"]),("rich_country",r"мы живем скорее в богатой",["в богатой"],["в бедной"]),("global_influence",r"влияние россии в мире в последние годы растет",["растет"],["снижается"]),("global_attitude",r"в последние годы отношение к россии в мире улучшается",["улучшается"],["ухудшается"]),("current_global_attitude",r"сегодня относятся к россии в мире",["скорее хорошо"],["скорее плохо"])]
def validation_spec(q):
    s=clean_text(q).lower(); return next(((m,p,n) for m,pat,p,n in VALIDATION_SPECS if re.search(pat,s)),None)
def parse_validation_workbook(df,sid=15305):
    pub=publication_date_for(sid); fw=(pd.NaT,pd.NaT,pd.NaT); rows=[]
    for rr in range(min(50,len(df))):
        txt=clean_text(df.iloc[rr,0])
        if "фомнибус" in txt.lower(): fw=parse_fieldwork_range(txt); break
    for r in range(len(df)):
        q=clean_text(df.iloc[r,0]); spec=validation_spec(q)
        if not spec: continue
        metric,pos_terms,neg_terms=spec; groups=header_for_question(df,r); ans=[]
        for rr in range(r+1,min(r+12,len(df))):
            lab=clean_text(df.iloc[rr,0]); n=pd.to_numeric(df.iloc[rr,1:],errors="coerce").notna().sum()
            if not lab:
                if ans: break
                continue
            if "?" in lab and n==0: break
            if n: ans.append((rr,lab.lower()))
        for c,gt,gname,gkey,_ in groups:
            pos=neg=0.; pf=nf=False
            for rr,lab in ans:
                v=pd.to_numeric(pd.Series([df.iloc[rr,c]]),errors="coerce").iloc[0]
                if pd.isna(v): continue
                if any(t in lab for t in pos_terms): pos+=float(v); pf=True
                if any(t in lab for t in neg_terms): neg+=float(v); nf=True
            if pf and nf: rows.append({"source_id":sid,"publication_date":pub,"fieldwork_start":fw[0],"fieldwork_end":fw[1],"wave_date":fw[2],"metric":metric,"question":q,"group_type":gt,"group":gname,"group_key":gkey,"positive":pos,"negative":neg,"balance":pos-neg})
    return pd.DataFrame(rows)
def valid_validation(x): return x is not None and not x.empty and set(x.metric)=={z[0] for z in VALIDATION_SPECS} and len(x[x.group_type.ne("overall")])>=150
def choose_validation(live_audit):
    attempts=[]
    for mode,p in workbook_candidates(15305,live_audit):
        try:
            frames=[parse_validation_workbook(df,15305) for df in read_workbook(p).values()]; x=pd.concat([z for z in frames if len(z)],ignore_index=True) if any(len(z) for z in frames) else pd.DataFrame(); ok=valid_validation(x); attempts.append({"source_id":15305,"block":"validation","mode":mode,"path":str(p),"rows":len(x),"status":"PASS" if ok else "REJECTED","sha256":sha256_file(p)})
            if ok: return x,pd.DataFrame(attempts)
        except Exception as e: attempts.append({"source_id":15305,"block":"validation","mode":mode,"path":str(p),"rows":0,"status":"ERROR","detail":repr(e)[:180]})
    return pd.DataFrame(),pd.DataFrame(attempts)
def bh_fdr(p):
    p=np.asarray(p,float); q=np.full(len(p),np.nan); ok=np.isfinite(p)
    if not ok.any(): return q
    ix=np.where(ok)[0]; order=ix[np.argsort(p[ix])]; m=len(order); vals=p[order]*m/np.arange(1,m+1); vals=np.minimum.accumulate(vals[::-1])[::-1]; q[order]=np.minimum(vals,1); return q


In [ ]:
# 4. QC И СТАТИСТИКА FIX4.1: EXACT/PERMUTATION, EXACT-DP CHANGE-POINT, BLOCKED VALIDATION
def qc_row(check,status,value="",detail=""): return {"check":check,"status":status,"value":value,"detail":detail}
def expectation_qc(exp,hist_points,hist_annual):
    req={"country_short","family_short","country_long","family_long"}; rows=[]; found=set(exp.construct.dropna()) if len(exp) else set(); rows.append(qc_row("exact_four_constructs","PASS" if found==req else "FAIL",str(sorted(found)))); rows.append(qc_row("no_missing_subject_horizon","PASS" if len(exp) and not exp[["subject","horizon"]].isna().any(axis=1).any() else "FAIL",int(exp[["subject","horizon"]].isna().any(axis=1).sum()) if len(exp) else -1)); core=exp[(exp.source_id.isin([15317,15337,15357,15380]))&(exp.group_type.eq("overall"))]; cells=core[["source_id","construct"]].drop_duplicates(); rows.append(qc_row("core_2026_16_of_16","PASS" if len(cells)==16 else "FAIL",len(cells))); maxerr=float((exp.response_sum-100).abs().max()) if len(exp) else np.nan; rows.append(qc_row("response_sum_near_100","PASS" if np.isfinite(maxerr) and maxerr<=CONFIG["response_sum_tolerance"] else "FAIL",maxerr)); rows.append(qc_row("core_fieldwork_dates","PASS" if len(core)==16 and core.wave_date.notna().all() else "FAIL",int(core.wave_date.isna().sum()))); cov=hist_annual.groupby("construct").year.nunique().to_dict() if len(hist_annual) else {}; rows.append(qc_row("history_2017_2026","PASS" if all(cov.get(c,0)>=10 for c in req) else "FAIL",cov)); anchors={(15317,"family_short"):30.73364177131527,(15317,"family_long"):32.46073298429319,(15380,"family_short"):19.287634408602152,(15380,"family_long"):25.721784776902887}
    for (sid,con),ref in anchors.items():
        q=core[(core.source_id.eq(sid))&(core.construct.eq(con))]; val=float(q.balance.iloc[0]) if len(q)==1 else np.nan; err=abs(val-ref) if np.isfinite(val) else np.inf; rows.append(qc_row(f"anchor_{sid}_{con}","PASS" if err<1e-6 else "FAIL",val,f"ref={ref:.12f}"))
    diffs=[]; hp=hist_points[(hist_points.year.eq(2026))&(hist_points.period_type.eq("wave"))] if len(hist_points) else pd.DataFrame()
    for _,r in core.iterrows():
        z=hp[hp.construct.eq(r.construct)].copy() if len(hp) else pd.DataFrame()
        if len(z) and pd.notna(r.wave_date): z["dd"]=(z.observation_date-r.wave_date).abs().dt.days; z=z.sort_values("dd"); diffs.append(abs(float(r.balance)-float(z.iloc[0].balance))) if len(z) and z.iloc[0].dd<=2 else None
    mx=max(diffs) if diffs else np.nan; rows.append(qc_row("html_binary_reconciliation","PASS" if len(diffs)==16 and mx<=CONFIG["html_binary_balance_tolerance"] else "FAIL",mx,f"matched={len(diffs)}/16")); return pd.DataFrame(rows)
def anxiety_qc(hist,current):
    ov=current[current.group_type.eq("overall")] if len(current) else pd.DataFrame(); cnt=ov.groupby("subject").size().to_dict() if len(ov) else {}; cov=hist.groupby("subject").year.nunique().to_dict() if len(hist) else {}; return pd.DataFrame([qc_row("anxiety_three_levels_2026","PASS" if set(ov.subject)=={"personal","country","world"} else "FAIL",sorted(set(ov.subject)) if len(ov) else []),qc_row("anxiety_items_2026","PASS" if all(cnt.get(s,0)>=10 for s in ["personal","country","world"]) else "FAIL",cnt),qc_row("anxiety_history_depth","PASS" if all(cov.get(s,0)>=10 for s in ["personal","country","world"]) else "FAIL",cov),qc_row("no_group_share_as_anxiety","PASS" if len(ov) and not ov.anxiety_item.str.lower().eq("доли групп").any() else "FAIL")])
def validation_qc(v):
    cnt=v.groupby("metric").size().to_dict() if len(v) else {}; return pd.DataFrame([qc_row("validation_five_metrics","PASS" if set(v.metric)=={z[0] for z in VALIDATION_SPECS} else "FAIL",cnt),qc_row("validation_group_rows","PASS" if len(v[v.group_type.ne("overall")])>=150 else "FAIL",len(v[v.group_type.ne("overall")]) if len(v) else 0)])
def overall_exp(exp): return exp[exp.group_type.eq("overall")].copy()
def _rank_corr_from_ranks(rx,ry):
    rx=np.asarray(rx,float); ry=np.asarray(ry,float); rx=rx-rx.mean(); ry=ry-ry.mean(); den=float(np.linalg.norm(rx)*np.linalg.norm(ry)); return float(np.dot(rx,ry)/den) if den>0 else np.nan

def permutation_spearman(x,y,permutations=None,exact_max_n=None,seed_offset=0):
    z=pd.DataFrame({"x":x,"y":y}).replace([np.inf,-np.inf],np.nan).dropna(); n=len(z); permutations=int(permutations or CONFIG["perm_n"]); exact_max_n=int(exact_max_n or CONFIG["exact_perm_max_n"])
    if n<3 or z.x.nunique()<2 or z.y.nunique()<2: return {"n":n,"rho":np.nan,"p_perm":np.nan,"p_asymptotic":np.nan,"perm_method":"NOT_TESTABLE","perm_n":0}
    rx=rankdata(z.x.to_numpy(),method="average"); ry=rankdata(z.y.to_numpy(),method="average"); obs=_rank_corr_from_ranks(rx,ry); pas=float(spearmanr(z.x,z.y).pvalue)
    eps=1e-12
    if n<=exact_max_n:
        hit=0; total=0
        for idx in itertools.permutations(range(n)):
            stat=_rank_corr_from_ranks(rx,ry[list(idx)]); hit+=int(abs(stat)>=abs(obs)-eps); total+=1
        p=hit/total; method="EXACT_PERMUTATION"; reps=total
    else:
        rng=np.random.default_rng(SEED+int(seed_offset)); hit=0
        for _ in range(permutations): hit+=int(abs(_rank_corr_from_ranks(rx,ry[rng.permutation(n)]))>=abs(obs)-eps)
        p=(hit+1)/(permutations+1); method="MONTE_CARLO_PERMUTATION"; reps=permutations
    return {"n":n,"rho":obs,"p_perm":float(p),"p_asymptotic":pas,"perm_method":method,"perm_n":reps}

def gap_tables(exp):
    ov=overall_exp(exp); mm=[]; hz=[]
    for (sid,h),g in ov.groupby(["source_id","horizon"]):
        p=g.pivot_table(index=["source_id","horizon","wave_date"],columns="subject",values="balance",aggfunc="first").reset_index()
        for _,r in p.dropna(subset=["country","family"]).iterrows() if {"country","family"}.issubset(p.columns) else []: mm.append({"source_id":sid,"horizon":h,"wave_date":r.wave_date,"country":r.country,"family":r.family,"micro_macro_gap":r.family-r.country})
    for (sid,s),g in ov.groupby(["source_id","subject"]):
        d={r.horizon:r for _,r in g.iterrows()}
        if "short" in d and "long" in d:
            a,b=d["short"],d["long"]; hz.append({"source_id":sid,"subject":s,"short_wave_date":a.wave_date,"long_wave_date":b.wave_date,"lag_days":(b.wave_date-a.wave_date).days if pd.notna(a.wave_date) and pd.notna(b.wave_date) else np.nan,"short_balance":a.balance,"long_balance":b.balance,"horizon_contrast":b.balance-a.balance,"independent_adjacent_waves":True})
    mm,hz=pd.DataFrame(mm),pd.DataFrame(hz); corr=[]; w=ov.pivot_table(index="source_id",columns="construct",values="balance",aggfunc="first")
    for j,(label,a,b) in enumerate([("macro_micro_short","country_short","family_short"),("macro_micro_long","country_long","family_long"),("horizon_country","country_short","country_long"),("horizon_family","family_short","family_long")]):
        z=w[[a,b]].dropna() if {a,b}.issubset(w.columns) else pd.DataFrame(); t=permutation_spearman(z[a],z[b],seed_offset=100+j) if len(z) else {"n":0,"rho":np.nan,"p_perm":np.nan,"p_asymptotic":np.nan,"perm_method":"NOT_TESTABLE","perm_n":0}; corr.append({"comparison":label,"n":t["n"],"spearman_rho":t["rho"],"p_value_perm":t["p_perm"],"p_value_asymptotic":t["p_asymptotic"],"test_method":t["perm_method"],"permutations":t["perm_n"]})
    corr=pd.DataFrame(corr); corr["q_fdr_perm"]=bh_fdr(corr.p_value_perm.to_numpy()) if len(corr) else []; corr["significant_fdr_05"]=corr.q_fdr_perm<.05 if len(corr) else []; return mm,hz,corr

def bootstrap_theilsen(g,B=None):
    B=B or CONFIG["bootstrap_n"]; g=g.dropna(subset=["year","balance"]).sort_values("year"); x=g.year.to_numpy(float); y=g.balance.to_numpy(float)
    if len(g)<CONFIG["min_trend_n"]: return np.nan,np.nan,np.nan,np.nan
    base=theilslopes(y,x,0.95); rng=np.random.default_rng(SEED); vals=[]
    for _ in range(B):
        ix=rng.integers(0,len(g),len(g)); xb,yb=x[ix],y[ix]
        if len(np.unique(xb))<2: continue
        try: vals.append(float(theilslopes(yb,xb,0.95).slope))
        except Exception: pass
    lo,hi=np.quantile(vals,[.025,.975]) if vals else (np.nan,np.nan); return float(base.slope),float(lo),float(hi),float(base.intercept)
def trend_analysis(annual):
    rows=[]
    for con,g in annual.groupby("construct"):
        slope,lo,hi,inter=bootstrap_theilsen(g); rows.append({"construct":con,"n":len(g),"year_min":int(g.year.min()),"year_max":int(g.year.max()),"slope_pp_per_year":slope,"ci_low":lo,"ci_high":hi,"intercept":inter,"direction":"up" if np.isfinite(lo) and lo>0 else ("down" if np.isfinite(hi) and hi<0 else "uncertain")})
    return pd.DataFrame(rows)
def exact_penalized_cp(z,pen,min_size=2):
    z=np.asarray(z,float); n=len(z); ps=np.r_[0.,np.cumsum(z)]; ps2=np.r_[0.,np.cumsum(z*z)]; F=np.full(n+1,np.inf); prev=np.full(n+1,-1,int); F[0]=-pen
    def cost(a,b):
        nn=b-a
        if nn<=0: return np.inf
        s=ps[b]-ps[a]; ss=ps2[b]-ps2[a]; return float(max(0.,ss-s*s/nn))
    for t in range(min_size,n+1):
        for s in range(0,t-min_size+1):
            if s>0 and (s<min_size or not np.isfinite(F[s])): continue
            v=F[s]+cost(s,t)+pen
            if v<F[t]: F[t]=v; prev[t]=s
    cps=[]; t=n
    while t>0 and prev[t]>=0:
        s=int(prev[t]); cps.append(s) if s>0 else None; t=s
    return sorted(cps),float(F[n])

def changepoint_analysis(annual):
    rows=[]; penalties=[.5,.75,1,1.25,1.5,2,3]
    for con,g in annual.groupby("construct"):
        g=g.dropna(subset=["balance"]).sort_values("year").reset_index(drop=True)
        if len(g)<CONFIG["min_changepoint_n"]: continue
        sd=float(g.balance.std(ddof=0)); z=((g.balance-g.balance.mean())/(sd if sd>0 else 1)).to_numpy(); hits=[]
        for mult in penalties:
            pen=float(mult*np.log(len(g))); cps,obj=exact_penalized_cp(z,pen,2)
            for cp in cps: hits.append({"construct":con,"engine":"EXACT_DP_L2","penalty_multiplier":mult,"penalty":pen,"objective":obj,"cp_index":cp,"cp_year":int(g.iloc[cp].year)})
        if hits:
            h=pd.DataFrame(hits); f=h.groupby(["construct","engine","cp_year"],as_index=False).agg(penalty_hits=("penalty_multiplier","nunique"),penalty_min=("penalty_multiplier","min"),penalty_max=("penalty_multiplier","max")); f["stability_share"]=f.penalty_hits/len(penalties); f["stable"]=f.stability_share>=.60; rows+=f.to_dict("records")
    return pd.DataFrame(rows)

def changepoint_analysis(annual):
    rows=[]; penalties=[.5,.75,1,1.25,1.5,2,3]
    for con,g in annual.groupby("construct"):
        g=g.dropna(subset=["balance"]).sort_values("year").reset_index(drop=True)
        if len(g)<CONFIG["min_changepoint_n"]: continue
        sd=float(g.balance.std(ddof=0)); z=((g.balance-g.balance.mean())/(sd if sd>0 else 1)).to_numpy(); hits=[]
        for mult in penalties:
            pen=float(mult*np.log(len(g))); cps,obj=exact_penalized_cp(z,pen,2)
            for cp in cps: hits.append({"construct":con,"engine":"EXACT_DP_L2","penalty_multiplier":mult,"penalty":pen,"objective":obj,"cp_index":cp,"cp_year":int(g.iloc[cp].year)})
        if hits:
            h=pd.DataFrame(hits); f=h.groupby(["construct","engine","cp_year"],as_index=False).agg(penalty_hits=("penalty_multiplier","nunique"),penalty_min=("penalty_multiplier","min"),penalty_max=("penalty_multiplier","max")); f["stability_share"]=f.penalty_hits/len(penalties); f["stable"]=f.stability_share>=.60; rows+=f.to_dict("records")
    return pd.DataFrame(rows)

def cluster_profiles(exp):
    x=exp[(exp.group_type.ne("overall"))&(exp.source_id.isin([15277,15317,15337,15357,15380]))&exp.balance.notna()].copy(); x["entity_key"]=x.group_type+"::"+x.group_key; x["feature"]=x.source_id.astype(str)+"__"+x.construct; raw=x.pivot_table(index=["entity_key","group_type","group_key"],columns="feature",values="balance",aggfunc="mean"); coverage=raw.notna().mean(axis=1); raw=raw.loc[coverage>=CONFIG["min_cluster_coverage"]]
    if len(raw)<CONFIG["min_cluster_groups"] or raw.shape[1]<8: return pd.DataFrame(),pd.DataFrame(),raw.reset_index(),pd.DataFrame(),pd.DataFrame(),pd.DataFrame(),pd.DataFrame(),np.empty((0,0))
    imputer=SimpleImputer(strategy="median"); X=imputer.fit_transform(raw); imputed_cells=int(raw.isna().sum().sum()); Xs=StandardScaler().fit_transform(X); sil=[]; best=None
    for k in range(2,min(7,len(raw))):
        lab=KMeans(n_clusters=k,n_init=50,random_state=SEED).fit_predict(Xs); sc=silhouette_score(Xs,lab); sil.append({"k":k,"silhouette":sc}); best=(sc,k,lab) if best is None or sc>best[0] else best
    bestsil,k,base=best; rng=np.random.default_rng(SEED); p=Xs.shape[1]; aris=[]
    for b in range(CONFIG["cluster_bootstrap_n"]): cols=rng.integers(0,p,p); aris.append(adjusted_rand_score(base,KMeans(n_clusters=k,n_init=20,random_state=SEED+b+1).fit_predict(Xs[:,cols])))
    q=np.quantile(aris,[.025,.5,.975]); cols=list(raw.columns); waves=sorted(set(c.split("__")[0] for c in cols)); loo=[]
    for w in waves:
        idx=[j for j,c in enumerate(cols) if not c.startswith(w+"__")]
        if len(idx)>=4: loo.append({"omitted_source":int(w),"ari":adjusted_rand_score(base,KMeans(n_clusters=k,n_init=50,random_state=SEED).fit_predict(Xs[:,idx]))})
    seed=[]
    for s in range(CONFIG["cluster_seed_n"]): seed.append({"seed":s,"ari":adjusted_rand_score(base,KMeans(n_clusters=k,n_init=20,random_state=s).fit_predict(Xs))})
    loo_mean=float(pd.DataFrame(loo).ari.mean()) if loo else np.nan; seed_mean=float(pd.DataFrame(seed).ari.mean()) if seed else np.nan; status="STABLE" if np.mean(aris)>=.75 and q[0]>=.50 and loo_mean>=.60 and seed_mean>=.90 else ("MODERATE" if np.mean(aris)>=.60 and loo_mean>=.40 else "UNSTABLE"); stab=pd.DataFrame([{"k":k,"silhouette":bestsil,"bootstrap_n":len(aris),"ari_mean":np.mean(aris),"ari_median":q[1],"ari_ci_low":q[0],"ari_ci_high":q[2],"loo_ari_mean":loo_mean,"seed_ari_mean":seed_mean,"status":status,"groups_n":len(raw),"features_n":raw.shape[1],"imputed_cells":imputed_cells,"coverage_threshold":CONFIG["min_cluster_coverage"]}]); ass=raw.reset_index()[["entity_key","group_type","group_key"]].copy(); ass["coverage"]=coverage.loc[raw.index].to_numpy(); ass["cluster"]=base; imp=pd.DataFrame(X,index=raw.index,columns=raw.columns).reset_index(); return ass,stab,raw.reset_index(),imp,pd.DataFrame(sil),pd.DataFrame(loo),pd.DataFrame(seed),Xs
def embedding_profiles(assignments,Xs):
    if assignments.empty or len(assignments)<3: return pd.DataFrame()
    p=PCA(n_components=2,random_state=SEED).fit_transform(Xs); out=assignments.copy(); out["pca1"],out["pca2"]=p[:,0],p[:,1]
    if HAS_UMAP and len(out)>=10:
        try: u=umap.UMAP(n_components=2,n_neighbors=min(10,len(out)-1),min_dist=.15,random_state=SEED).fit_transform(Xs); out["umap1"],out["umap2"]=u[:,0],u[:,1]
        except Exception: pass
    return out
def anxiety_combined(hist,current):
    cur=current[current.group_type.eq("overall")].copy(); cur["observation_date"]=cur.wave_date; cur["year"]=cur.wave_date.dt.year; return pd.concat([hist[["source_id","subject","anxiety_item","item_key","observation_date","year","value"]],cur[["source_id","subject","anxiety_item","item_key","observation_date","year","value"]]],ignore_index=True)
def anxiety_trends(anx):
    rows=[]
    for (sub,key),g in anx.groupby(["subject","item_key"]):
        g=g.sort_values("year").drop_duplicates("year",keep="last")
        if len(g)<6: continue
        z=theilslopes(g.value.to_numpy(float),g.year.to_numpy(float),.95); rows.append({"subject":sub,"item_key":key,"item":g.anxiety_item.iloc[-1],"n":len(g),"year_min":int(g.year.min()),"year_max":int(g.year.max()),"slope_pp_per_year":float(z.slope),"ci_low":float(z.low_slope),"ci_high":float(z.high_slope)})
    return pd.DataFrame(rows)
def _stratified_rank_stat(m):
    xs=[]; ys=[]; used=[]
    for gt,g in m.groupby("group_type"):
        g=g.dropna(subset=["balance_val","balance_exp"])
        if len(g)<CONFIG["validation_min_block_n"] or g.balance_val.nunique()<2 or g.balance_exp.nunique()<2: continue
        rx=rankdata(g.balance_val.to_numpy(),method="average"); ry=rankdata(g.balance_exp.to_numpy(),method="average"); rx=(rx-rx.mean())/(rx.std(ddof=0) or 1); ry=(ry-ry.mean())/(ry.std(ddof=0) or 1); xs.extend(rx); ys.extend(ry); used.append(gt)
    return (_rank_corr_from_ranks(np.asarray(xs),np.asarray(ys)) if len(xs)>=6 else np.nan),used

def _stratified_perm_test(m,seed_offset=0):
    obs,used=_stratified_rank_stat(m); blocks=[]
    for gt,g in m[m.group_type.isin(used)].groupby("group_type"):
        g=g.dropna(subset=["balance_val","balance_exp"]); rx=rankdata(g.balance_val.to_numpy(),method="average"); ry=rankdata(g.balance_exp.to_numpy(),method="average"); rx=(rx-rx.mean())/(rx.std(ddof=0) or 1); ry=(ry-ry.mean())/(ry.std(ddof=0) or 1); blocks.append((gt,rx,ry))
    if len(blocks)<CONFIG["validation_min_blocks"] or not np.isfinite(obs): return obs,np.nan,0,"NOT_TESTABLE",used
    rng=np.random.default_rng(SEED+5000+seed_offset); B=int(CONFIG["perm_n"]); hit=0; eps=1e-12
    for _ in range(B):
        xx=[]; yy=[]
        for gt,rx,ry in blocks: xx.extend(rx); yy.extend(ry[rng.permutation(len(ry))])
        hit+=int(abs(_rank_corr_from_ranks(np.asarray(xx),np.asarray(yy)))>=abs(obs)-eps)
    return obs,(hit+1)/(B+1),B,"STRATIFIED_WITHIN_BLOCK_MC_PERMUTATION",used

def validation_correlations(validation,exp,sid=None):
    sid=sid or CONFIG["validation_expectation_source"]; e=exp[(exp.source_id.eq(sid))&(exp.group_type.ne("overall"))].copy(); v=validation[validation.group_type.ne("overall")].copy(); rows=[]; block_rows=[]; test_id=0
    for metric,vg in v.groupby("metric"):
        for con,eg in e.groupby("construct"):
            m=vg[["group_type","group_key","balance","wave_date"]].merge(eg[["group_type","group_key","balance","wave_date"]],on=["group_type","group_key"],suffixes=("_val","_exp")); test_id+=1
            for gt,b in m.groupby("group_type"):
                b=b.dropna(subset=["balance_val","balance_exp"]); rho=float(spearmanr(b.balance_val,b.balance_exp).statistic) if len(b)>=3 and b.balance_val.nunique()>1 and b.balance_exp.nunique()>1 else np.nan; block_rows.append({"validation_metric":metric,"expectation_construct":con,"group_type":gt,"n":len(b),"spearman_rho":rho})
            stat,p,B,method,used=_stratified_perm_test(m,seed_offset=test_id); bdf=pd.DataFrame([r for r in block_rows if r["validation_metric"]==metric and r["expectation_construct"]==con]); finite=bdf.spearman_rho.dropna() if len(bdf) else pd.Series(dtype=float); sign_cons=float(max((finite>0).mean(),(finite<0).mean())) if len(finite) else np.nan; lag=float(np.nanmedian((m.wave_date_exp-m.wave_date_val).dt.days)) if len(m) else np.nan
            # leave-one-block-out statistic
            loo=[]
            for gt in used:
                st,_=_stratified_rank_stat(m[m.group_type.ne(gt)]); loo.append(st)
            rows.append({"validation_metric":metric,"expectation_construct":con,"matched_n":len(m),"group_types_n":len(used),"stratified_rho":stat,"p_value_perm":p,"test_method":method,"permutations":B,"sign_consistency":sign_cons,"loo_rho_min":float(np.nanmin(loo)) if len(loo) and np.isfinite(loo).any() else np.nan,"loo_rho_max":float(np.nanmax(loo)) if len(loo) and np.isfinite(loo).any() else np.nan,"fieldwork_lag_days":lag})
    out=pd.DataFrame(rows); blocks=pd.DataFrame(block_rows); out["q_fdr_perm"]=bh_fdr(out.p_value_perm.to_numpy()) if len(out) else []; out["significant_fdr_05"]=out.q_fdr_perm<.05 if len(out) else []; out["robust_fdr_05"]=(out.significant_fdr_05&(out.group_types_n>=CONFIG["validation_min_blocks"])&(out.sign_consistency>=CONFIG["validation_sign_consistency"])) if len(out) else []; return out,blocks

def cross_component_correlations(annual,anx,current):
    top=current[current.group_type.eq("overall")].copy(); top=top[~top.anxiety_item.str.lower().str.contains("затрудня|ничего не вызывает",regex=True)]; keys=[]
    for sub,g in top.groupby("subject"): keys += [(sub,k) for k in g.nlargest(5,"value").item_key.tolist()]
    rows=[]; j=0
    for sub,key in keys:
        a=anx[(anx.subject.eq(sub))&(anx.item_key.eq(key))].sort_values("year").drop_duplicates("year",keep="last")[["year","value","anxiety_item"]]
        for con,e in annual.groupby("construct"):
            m=e[["year","balance"]].merge(a,on="year"); j+=1; t=permutation_spearman(m.balance,m.value,permutations=CONFIG["cross_perm_n"],exact_max_n=CONFIG["exact_perm_max_n"],seed_offset=9000+j) if len(m)>=3 else {"n":len(m),"rho":np.nan,"p_perm":np.nan,"p_asymptotic":np.nan,"perm_method":"NOT_TESTABLE","perm_n":0}; rows.append({"anxiety_subject":sub,"anxiety_item":a.anxiety_item.iloc[-1] if len(a) else key,"expectation_construct":con,"n":t["n"],"spearman_rho":t["rho"],"p_value_perm":t["p_perm"],"p_value_asymptotic":t["p_asymptotic"],"test_method":t["perm_method"],"permutations":t["perm_n"]})
    out=pd.DataFrame(rows); out["q_fdr_perm"]=bh_fdr(out.p_value_perm.to_numpy()) if len(out) else []; out["significant_fdr_05"]=out.q_fdr_perm<.05 if len(out) else []; return out


def acceptance_checklist(exp,hist_annual,anx_hist,anx_cur,gapcorr,cps,clusters,stab,vc,vblocks,cross,source_audit):
    req={"country_short","family_short","country_long","family_long"}; rows=[]
    def add(check,ok,detail=""): rows.append({"check":check,"status":"PASS" if bool(ok) else "FAIL","detail":detail})
    found=set(exp.construct.dropna().unique()) if len(exp) else set(); add("four_expectation_constructs",found==req,str(sorted(found))); core=exp[(exp.source_id.isin([15317,15337,15357,15380]))&(exp.group_type.eq("overall"))]; add("core_2026_16_of_16",len(core)==16,len(core)); cov=hist_annual.groupby("construct").year.nunique().to_dict() if len(hist_annual) else {}; add("history_2017_2026_all_constructs",all(cov.get(c,0)>=10 for c in req),str(cov)); add("three_anxiety_levels",set(anx_cur.subject.unique())=={"personal","country","world"} and set(anx_hist.subject.unique())=={"personal","country","world"},str(sorted(set(anx_cur.subject.unique())))); add("small_n_permutation_tests",len(gapcorr)==4 and gapcorr.test_method.str.contains("PERMUTATION").all(),gapcorr[["comparison","n","test_method"]].to_dict("records") if len(gapcorr) else "none"); add("changepoint_method_exact_dp",len(cps)==0 or set(cps.engine.unique())=={"EXACT_DP_L2"},str(sorted(cps.engine.unique())) if len(cps) else "no cps"); add("cluster_no_imputation",len(stab)>0 and int(stab.imputed_cells.iloc[0])==0,int(stab.imputed_cells.iloc[0]) if len(stab) else -1); add("cluster_not_unstable",len(stab)>0 and stab.status.iloc[0] in ["STABLE","MODERATE"],stab.status.iloc[0] if len(stab) else "none"); add("blocked_validation",len(vc)>0 and vc.test_method.str.contains("STRATIFIED").all() and len(vblocks)>0,f"tests={len(vc)}, blocks={len(vblocks)}"); add("cross_component_permutation_fdr",len(cross)>0 and "q_fdr_perm" in cross.columns,f"tests={len(cross)}"); add("source_provenance",len(source_audit)>0 and source_audit.status.eq("PASS").any(),source_audit[source_audit.status.eq("PASS")][["block","mode"]].to_dict("records") if len(source_audit) else "none"); return pd.DataFrame(rows)


In [ ]:
# 5. УСЛОВНЫЙ ML/SHAP, ВИЗУАЛИЗАЦИЯ И ЭКСПОРТ FIX4.1
EXPECT_KEYS=["полгода","пол года","3 5 лет","три пять","через три","жизнь вашей семьи","жизнь в нашей стране"]; ID_KEYS=["id","номер","respondent","респондент","case","анкета"]
def raw_micro_candidates():
    out=[]; reg=[]
    if not HAS_PYREADSTAT: return out,pd.DataFrame([{"status":"PYREADSTAT_UNAVAILABLE"}])
    for p in LIVE.rglob("*.sav"):
        try: df,md=pyreadstat.read_sav(str(p),apply_value_formats=True); labels=dict(zip(md.column_names,md.column_labels or md.column_names)); out.append((p.name,df,labels)); reg.append({"file":str(p),"rows":len(df),"cols":df.shape[1]})
        except Exception as e: reg.append({"file":str(p),"rows":0,"cols":0,"error":repr(e)[:160]})
    return out,pd.DataFrame(reg)
def score_expectation_series(s):
    def f(v):
        z=norm_token(v); return 1. if any(k in z for k in ["лучше","улучш"]) else (-1. if any(k in z for k in ["хуже","ухудш"]) else (0. if any(k in z for k in ["не измен","без измен","не помен"]) else np.nan))
    return s.map(f)
def bootstrap_metric_ci(y,pred,B=400):
    rng=np.random.default_rng(SEED); y,pred=np.asarray(y),np.asarray(pred); vals=[]
    for _ in range(B):
        ix=rng.integers(0,len(y),len(y))
        try: vals.append(f1_score(y[ix],pred[ix],average="macro"))
        except Exception: pass
    return tuple(np.quantile(vals,[.025,.975])) if vals else (np.nan,np.nan)
def run_ml(candidates):
    empty=(pd.DataFrame(),pd.DataFrame(),pd.DataFrame(),pd.DataFrame(),{"status":"SKIPPED_NO_MICRODATA"})
    if not CONFIG["run_ml"] or not candidates: return empty
    scored=[]
    for key,df,labels in candidates:
        tc=[c for c in df.columns if any(k in norm_token(labels.get(c,c)) for k in EXPECT_KEYS)]; scored.append((len(tc),len(df),key,df,labels,tc))
    scored.sort(reverse=True,key=lambda z:(z[0],z[1])); chosen=scored[0] if scored and scored[0][0]>=2 else None
    if chosen is None: return empty
    nt,n,key,df,labels,target_cols=chosen; scores=pd.DataFrame({str(c):score_expectation_series(df[c]) for c in target_cols}); A=scores.mean(axis=1,skipna=True); valid=scores.notna().sum(axis=1)>=2; y=pd.Series(np.where(A>.25,"positive",np.where(A<-.25,"negative","stable")),index=df.index).loc[valid]; work=df.loc[valid].copy(); counts=y.value_counts()
    if len(work)<CONFIG["min_micro_n"] or len(counts)<2 or counts.min()<CONFIG["min_class_n"]: return empty[:-1]+({"status":"SKIPPED_CLASS_SUPPORT","dataset":key,"n":len(work),"classes":counts.to_dict()},)
    exclude=set(target_cols)
    for c in work.columns:
        if any(k in norm_token(labels.get(c,c)) for k in EXPECT_KEYS+ID_KEYS): exclude.add(c)
    X=work[[c for c in work.columns if c not in exclude]].copy(); X=X.loc[:,X.notna().mean()>=.2]; X=X.loc[:,[c for c in X if X[c].nunique(dropna=True)>1]]
    if X.shape[1]<2: return empty[:-1]+({"status":"SKIPPED_NO_FEATURES","dataset":key},)
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=CONFIG["test_size"],random_state=SEED,stratify=y); cat=[c for c in X if X[c].dtype==object or X[c].nunique(dropna=True)<=20]; num=[c for c in X if c not in cat]; pre=ColumnTransformer([("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),num),("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",OneHotEncoder(handle_unknown="ignore",min_frequency=5))]),cat)]); models=[("LogisticRegression",Pipeline([("pre",pre),("model",LogisticRegression(max_iter=2500,class_weight="balanced",random_state=SEED))]))]
    if HAS_LGBM: models.append(("LightGBM",Pipeline([("pre",pre),("model",LGBMClassifier(n_estimators=350,learning_rate=.035,num_leaves=20,class_weight="balanced",random_state=SEED,verbosity=-1))])))
    metrics=[]; preds=[]; fi=pd.DataFrame(); shap_df=pd.DataFrame()
    for name,m in models:
        try: m.fit(Xtr,ytr); pr=m.predict(Xte); lo,hi=bootstrap_metric_ci(yte.to_numpy(),pr); metrics.append({"model":name,"macro_f1":f1_score(yte,pr,average="macro"),"f1_ci_low":lo,"f1_ci_high":hi,"balanced_accuracy":balanced_accuracy_score(yte,pr),"n_train":len(Xtr),"n_test":len(Xte)}); preds += [{"model":name,"row_index":i,"y_true":a,"y_pred":b} for i,a,b in zip(yte.index,yte,pr)]
        except Exception as e: metrics.append({"model":name,"error":repr(e)[:180]})
    if HAS_CATBOOST:
        try:
            Xc=X.copy(); [Xc.__setitem__(c,Xc[c].astype(str).fillna("NA")) for c in cat]
            for c in num: z=pd.to_numeric(Xc[c],errors="coerce"); Xc[c]=z.fillna(z.median())
            xa,xb,ya,yb=train_test_split(Xc,y,test_size=CONFIG["test_size"],random_state=SEED,stratify=y); ci=[Xc.columns.get_loc(c) for c in cat]; cb=CatBoostClassifier(iterations=450,depth=6,learning_rate=.035,loss_function="MultiClass",verbose=False,random_seed=SEED,auto_class_weights="Balanced",allow_writing_files=False); cb.fit(xa,ya,cat_features=ci); pr=cb.predict(xb).ravel(); lo,hi=bootstrap_metric_ci(yb.to_numpy(),pr); metrics.append({"model":"CatBoost","macro_f1":f1_score(yb,pr,average="macro"),"f1_ci_low":lo,"f1_ci_high":hi,"balanced_accuracy":balanced_accuracy_score(yb,pr),"n_train":len(xa),"n_test":len(xb)}); fi=pd.DataFrame({"feature":Xc.columns,"importance":cb.get_feature_importance()}).sort_values("importance",ascending=False)
            if CONFIG["run_shap"] and HAS_SHAP:
                sample=xb.sample(min(300,len(xb)),random_state=SEED); arr=np.asarray(shap.TreeExplainer(cb).shap_values(sample)); ma=np.abs(arr).mean(axis=(0,2)) if arr.ndim==3 and arr.shape[1]==sample.shape[1] else (np.abs(arr).mean(axis=0) if arr.ndim==2 else None); shap_df=pd.DataFrame({"feature":sample.columns,"mean_abs_shap":ma}).sort_values("mean_abs_shap",ascending=False) if ma is not None and len(ma)==sample.shape[1] else pd.DataFrame()
        except Exception as e: metrics.append({"model":"CatBoost","error":repr(e)[:180]})
    return pd.DataFrame(metrics),pd.DataFrame(preds),fi,shap_df,{"status":"OK","dataset":key,"n":len(work),"target_columns":len(target_cols),"features":X.shape[1],"class_counts":counts.to_dict()}
def auto_width_excel(writer,max_width=42):
    if writer.engine!="xlsxwriter": return
    for ws in writer.book.worksheets():
        rm,cm=ws.dim_rowmax,ws.dim_colmax
        if rm is None or cm is None: continue
        ws.freeze_panes(1,0); ws.autofilter(0,0,max(0,rm),cm)
        for j in range(min(cm+1,90)): ws.set_column(j,j,min(max_width,18))
def write_excel(path,sheets):
    engine="xlsxwriter" if importlib.util.find_spec("xlsxwriter") else "openpyxl"
    with pd.ExcelWriter(path,engine=engine,datetime_format="yyyy-mm-dd") as w:
        for name,df in sheets.items(): (df if df is not None else pd.DataFrame()).to_excel(w,re.sub(r"[\\/*?:\[\]]","_",name)[:31],index=False)
        auto_width_excel(w)
def build_canonical_long(exp,hp,ac,ah,val):
    parts=[]
    if len(exp): idc=["source_id","publication_date","fieldwork_start","fieldwork_end","wave_date","question","subject","horizon","construct","group_type","group","group_key"]; z=exp[idc+["better","worse","same","dk"]].melt(id_vars=idc,var_name="response_category",value_name="value"); z["dataset"]="expectations_group"; parts.append(z)
    if len(hp): z=hp.copy(); z["dataset"]="expectations_history"; z["group_type"]="overall"; z["group"]="Все опрошенные"; z["group_key"]="все опрошенные"; z=z.rename(columns={"observation_date":"wave_date"}); parts.append(z)
    if len(ac): z=ac.copy(); z["dataset"]="anxiety_current"; z["response_category"]=z.anxiety_item; parts.append(z)
    if len(ah): z=ah.copy(); z["dataset"]="anxiety_history"; z["group_type"]="overall"; z["group"]="Все опрошенные"; z["group_key"]="все опрошенные"; z["response_category"]=z.anxiety_item; z=z.rename(columns={"observation_date":"wave_date"}); parts.append(z)
    if len(val): z=val.copy(); z["dataset"]="validation"; z["response_category"]="balance"; z["value"]=z.balance; parts.append(z)
    cols=sorted(set().union(*[set(x.columns) for x in parts])) if parts else []; return pd.concat([x.reindex(columns=cols) for x in parts],ignore_index=True) if parts else pd.DataFrame()
def save_figures(annual,exp,mm,hz,anx,ac,clusters,emb,vc,cross,cps,ml,shap_df):
    made=[]; plt.rcParams.update({"figure.dpi":110,"savefig.dpi":180,"font.size":10})
    def save(name): p=FIG/name; plt.tight_layout(); plt.savefig(p,bbox_inches="tight"); plt.close(); made.append(name)
    if len(annual):
        plt.figure(figsize=(10,5.6)); [plt.plot(g.year,g.balance,marker="o",label=con) for con,g in annual.groupby("construct")]; [plt.axvline(r.cp_year,ls="--",lw=.8,alpha=.3) for _,r in cps[cps.stable.eq(True)].iterrows()] if len(cps) and "stable" in cps else None; plt.axhline(0,lw=.8); plt.xlabel("Год"); plt.ylabel("Баланс, п.п."); plt.title("Годовая динамика ожиданий, 2017–2026"); plt.legend(); save("01_expectations_annual.png")
    core=overall_exp(exp[exp.source_id.isin([15277,15317,15337,15357,15380])])
    if len(core): plt.figure(figsize=(10,5.6)); [plt.plot(g.wave_date,g.balance,marker="o",label=con) for con,g in core.groupby("construct")]; plt.axhline(0,lw=.8); plt.ylabel("Баланс, п.п."); plt.title("Ожидания: полевые волны 2025–2026"); plt.legend(); save("02_expectations_waves.png")
    if len(mm): plt.figure(figsize=(9,5)); [plt.plot(g.wave_date,g.micro_macro_gap,marker="o",label=h) for h,g in mm.groupby("horizon")]; plt.axhline(0,lw=.8); plt.ylabel("Семья − страна, п.п."); plt.title("Микро–макро разрыв"); plt.legend(); save("03_micro_macro_gap.png")
    if len(hz): plt.figure(figsize=(9,5)); [plt.plot(g.short_wave_date,g.horizon_contrast,marker="o",label=s) for s,g in hz.groupby("subject")]; plt.axhline(0,lw=.8); plt.ylabel("Long − short, п.п."); plt.title("Контраст горизонтов: соседние независимые волны"); plt.legend(); save("04_horizon_contrast.png")
    cur=ac[ac.group_type.eq("overall")]
    for sub,g in cur.groupby("subject"):
        q=g[~g.anxiety_item.str.lower().str.contains("затрудня|ничего не вызывает",regex=True)].nlargest(10,"value").sort_values("value"); plt.figure(figsize=(9,5.4)); plt.barh(range(len(q)),q.value); plt.yticks(range(len(q)),[clean_text(x)[:68] for x in q.anxiety_item]); plt.xlabel("Доля, %"); plt.title(f"Тревоги 2026: {sub}"); save(f"05_anxiety_{sub}_2026.png")
    for sub,g in anx.groupby("subject"):
        latest=g.sort_values("year").groupby("item_key",as_index=False).tail(1).nlargest(5,"value").item_key.tolist(); q=g[g.item_key.isin(latest)]; plt.figure(figsize=(10,5.5)); [plt.plot(h.year,h.value,marker="o",label=clean_text(h.anxiety_item.iloc[-1])[:43]) for key,h in q.groupby("item_key")]; plt.ylabel("Доля, %"); plt.title(f"Динамика ключевых тревог: {sub}"); plt.legend(fontsize=7); save(f"06_anxiety_trends_{sub}.png")
    if len(clusters): q=clusters.cluster.value_counts().sort_index(); plt.figure(figsize=(7,4.4)); plt.bar(q.index.astype(str),q.values); plt.xlabel("Кластер"); plt.ylabel("Число групп"); plt.title("Размеры кластеров социальных групп"); save("07_cluster_sizes.png")
    if len(emb): plt.figure(figsize=(7,5.2)); [plt.scatter(g.pca1,g.pca2,label=f"cluster {cl}",alpha=.8) for cl,g in emb.groupby("cluster")]; plt.xlabel("PCA 1"); plt.ylabel("PCA 2"); plt.title("PCA-проекция групп"); plt.legend(); save("08_group_pca.png")
    if len(vc): pv=vc.pivot(index="validation_metric",columns="expectation_construct",values="stratified_rho"); plt.figure(figsize=(8,5)); plt.imshow(pv.to_numpy(),aspect="auto",vmin=-1,vmax=1); plt.xticks(range(len(pv.columns)),pv.columns,rotation=35,ha="right"); plt.yticks(range(len(pv.index)),pv.index); plt.colorbar(label="Стратифицированная ранговая связь"); plt.title("Внешняя валидизация: within-group-type permutation"); save("09_external_validation_blocked.png")
    if len(cross): q=cross.sort_values("q_fdr_perm").head(20).copy(); plt.figure(figsize=(9,6)); plt.barh(range(len(q)),q.spearman_rho); plt.yticks(range(len(q)),[(clean_text(a)[:34]+" / "+b) for a,b in zip(q.anxiety_item,q.expectation_construct)],fontsize=7); plt.axvline(0,lw=.8); plt.xlabel("Spearman ρ"); plt.title("Ожидания ↔ тревоги: permutation + FDR"); save("10_cross_component.png")
    if len(ml) and "macro_f1" in ml.columns:
        q=ml.dropna(subset=["macro_f1"])
        if len(q): plt.figure(figsize=(7,4.4)); plt.bar(q.model,q.macro_f1); plt.ylim(0,1); plt.ylabel("Macro-F1"); plt.title("ML: качество моделей"); save("11_ml_f1.png")
    if len(shap_df): q=shap_df.head(15).sort_values("mean_abs_shap"); plt.figure(figsize=(8,5.4)); plt.barh(q.feature.astype(str),q.mean_abs_shap); plt.xlabel("Mean |SHAP|"); plt.title("SHAP"); save("12_shap.png")
    return made

def markdown_report(audit,qc,acceptance,gapcorr,trends,cps,stab,vc,vblocks,cross,ml):
    L=["# Итог вычислительного эксперимента FIX4.1 FINAL","",f"**Статус:** `{audit['status']}`",f"**Приемочный уровень:** `{audit.get('acceptance_level','')}`","","## Основные счетчики",""]+[f"- **{k}:** {v}" for k,v in audit.items()]
    for title,df in [("Приемочный чек-лист",acceptance),("Контроль качества",qc),("Макро/микро и горизонты: exact/permutation",gapcorr),("Theil–Sen",trends),("Change-point: EXACT_DP_L2",cps),("Устойчивость кластеров",stab),("Внешняя валидизация: стратифицированный permutation",vc),("Внешняя валидизация по типам групп",vblocks),("Ожидания ↔ тревоги: permutation + FDR",cross),("ML",ml)]: L += ["",f"## {title}","",("Нет данных для расчета." if df is None or df.empty else df.head(120).to_markdown(index=False))]
    L += ["","## Правила интерпретации","","- Микро–макро разрыв относится к одной полевой волне; p-value для малых n рассчитывается exact permutation.","- Контраст short/long относится к соседним независимым волнам и не является панельным эффектом.","- Theil–Sen трактуется как монотонный тренд только если bootstrap 95% CI не включает 0.","- Change-point — описательный структурный сдвиг по 10 годовым точкам; метод: точное penalized dynamic programming L2 (`EXACT_DP_L2`), не PELT.","- Тревоги — множественный выбор; доли не образуют единый суммарный индекс.","- Кластеры — предварительная типология опубликованных социальных срезов; интерпретируются только вместе с bootstrap/LOO/seed ARI.","- Внешняя валидизация использует перестановки только внутри `group_type`; итоговая robust-метка дополнительно требует согласованности направлений между блоками.","- ML/SHAP не запускается на агрегированных таблицах.","- Источник каждого обязательного блока фиксируется как LIVE или EMBEDDED_OFFICIAL_SNAPSHOT."]; return "\n".join(L)



def result_file_manifest(root=RESULTS):
    rows=[]
    for p in sorted(root.rglob("*")):
        if p.is_file(): rows.append({"path":str(p.relative_to(root)),"bytes":p.stat().st_size,"sha256":sha256_file(p)})
    return pd.DataFrame(rows)

def zip_audit(research_zip,full_zip):
    def hashes(zp,prefix=""):
        out={}
        with zipfile.ZipFile(zp) as z:
            for n in z.namelist():
                if n.endswith("/"): continue
                key=n[len(prefix):] if prefix and n.startswith(prefix) else n; out[key]=hashlib.sha256(z.read(n)).hexdigest()
        return out
    rh=hashes(research_zip); fh=hashes(full_zip,"results/"); common=set(rh)&set(fh); mismatch=[k for k in common if rh[k]!=fh[k]]; return pd.DataFrame([{"research_members":len(rh),"full_result_members":len(fh),"common_members":len(common),"missing_from_full_results":len(set(rh)-set(fh)),"hash_mismatches":len(mismatch),"research_zip_sha256":sha256_file(research_zip),"full_zip_sha256":sha256_file(full_zip),"status":"PASS" if len(set(rh)-set(fh))==0 and len(mismatch)==0 else "FAIL"}])

## 3. Единый запуск FIX4.1 FINAL

Одна ячейка выполняет полный цикл: materialize snapshot → live-загрузка → parse-aware fallback → QC → статистика → exact/permutation tests → exact-DP change-point → кластерная устойчивость → blocked external validation → условный ML/SHAP → графики → Excel/CSV/Parquet/Markdown → manifest → полный и исследовательский ZIP → проверка целостности пакетов.

In [ ]:
# 6. ЕДИНЫЙ ЗАПУСК FIX4.1 FINAL — один progress bar, без мусора
from IPython.display import display
BAR="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}"; progress=tqdm(total=100,desc="НИР — глава 1 FIX4.1",bar_format=BAR,dynamic_ncols=True,leave=True)
def tick(n,label): progress.set_postfix_str(label); progress.update(n)
def fail_gate(message,qc=None,source_audit=None):
    diag=pd.DataFrame([{"status":"FAILED_GATE","message":message}]); write_excel(RESULTS/"EXPERIMENT_AUDIT_FIX41.xlsx",{"audit":diag,"quality_checks":qc if qc is not None else pd.DataFrame(),"source_audit":source_audit if source_audit is not None else pd.DataFrame()}); progress.close(); raise RuntimeError(message)
SNAPSHOT_AUDIT=materialize_snapshot(); tick(4,"snapshot"); LIVE_AUDIT=fetch_live_parallel(); tick(8,"live / fallback")
exp_frames=[]; parse_audits=[]
for sid in [15277,15317,15337,15357,15380]: x,a=choose_expectation_source(sid,LIVE_AUDIT); exp_frames.append(x) if len(x) else None; parse_audits.append(a)
EXPECTATIONS=pd.concat(exp_frames,ignore_index=True) if exp_frames else pd.DataFrame(); EXPECT_PARSE_AUDIT=pd.concat(parse_audits,ignore_index=True) if parse_audits else pd.DataFrame(); tick(12,"ожидания: таблицы")
if EXPECTATIONS.empty: fail_gate("Не удалось восстановить структурные таблицы ожиданий даже из встроенного snapshot.",source_audit=EXPECT_PARSE_AUDIT)
EXPECT_HISTORY_LONG,EXPECT_HISTORY_POINTS,EXPECT_ANNUAL,HISTORY_AUDIT=choose_expectation_history(LIVE_AUDIT); tick(7,"ожидания: история")
if EXPECT_ANNUAL.empty: fail_gate("Не удалось восстановить длинную динамику ожиданий.",source_audit=HISTORY_AUDIT)
ANXIETY_CURRENT,ANX_CUR_AUDIT=choose_anxiety_current(LIVE_AUDIT); ANXIETY_HISTORY,ANX_HIST_AUDIT=choose_anxiety_history(LIVE_AUDIT); ANXIETY_COMBINED=anxiety_combined(ANXIETY_HISTORY,ANXIETY_CURRENT) if len(ANXIETY_CURRENT) and len(ANXIETY_HISTORY) else pd.DataFrame(); tick(9,"тревоги")
VALIDATION,VALIDATION_AUDIT=choose_validation(LIVE_AUDIT); tick(7,"внешняя валидизация: данные")
SOURCE_AUDIT=pd.concat([EXPECT_PARSE_AUDIT,HISTORY_AUDIT,ANX_CUR_AUDIT,ANX_HIST_AUDIT,VALIDATION_AUDIT],ignore_index=True); QC_EXP=expectation_qc(EXPECTATIONS,EXPECT_HISTORY_POINTS,EXPECT_ANNUAL); QC_ANX=anxiety_qc(ANXIETY_HISTORY,ANXIETY_CURRENT); QC_VAL=validation_qc(VALIDATION); QC=pd.concat([QC_EXP,QC_ANX,QC_VAL],ignore_index=True); tick(5,"QC")
if (QC.status=="FAIL").any(): fail_gate("Провален обязательный содержательный QC FIX4.1.",QC,SOURCE_AUDIT)
MM_GAPS,HORIZON_CONTRASTS,GAP_CORR=gap_tables(EXPECTATIONS); TRENDS=trend_analysis(EXPECT_ANNUAL); CHANGEPOINTS=changepoint_analysis(EXPECT_ANNUAL); ANXIETY_TRENDS=anxiety_trends(ANXIETY_COMBINED); tick(12,"exact/permutation + trends + DP")
CLUSTERS,CLUSTER_STABILITY,GROUP_MATRIX_RAW,GROUP_MATRIX_IMPUTED,SILHOUETTE,CLUSTER_LOO,CLUSTER_SEEDS,XS=cluster_profiles(EXPECTATIONS); EMBEDDING=embedding_profiles(CLUSTERS,XS) if len(CLUSTERS) else pd.DataFrame(); tick(9,"кластеры")
VALIDATION_CORR,VALIDATION_BY_BLOCK=validation_correlations(VALIDATION,EXPECTATIONS); CROSS_COMPONENT=cross_component_correlations(EXPECT_ANNUAL,ANXIETY_COMBINED,ANXIETY_CURRENT); tick(12,"blocked validation / связи")
MICRO_CANDIDATES,MICRO_REGISTRY=raw_micro_candidates(); ML_METRICS,ML_PREDICTIONS,FEATURE_IMPORTANCE,SHAP_IMPORTANCE,ML_STATUS=run_ml(MICRO_CANDIDATES); tick(4,"ML / SHAP")
ACCEPTANCE_CHECKLIST=acceptance_checklist(EXPECTATIONS,EXPECT_ANNUAL,ANXIETY_HISTORY,ANXIETY_CURRENT,GAP_CORR,CHANGEPOINTS,CLUSTERS,CLUSTER_STABILITY,VALIDATION_CORR,VALIDATION_BY_BLOCK,CROSS_COMPONENT,SOURCE_AUDIT)
if (ACCEPTANCE_CHECKLIST.status=="FAIL").any(): fail_gate("Провален финальный приемочный чек-лист FIX4.1.",pd.concat([QC,ACCEPTANCE_CHECKLIST.rename(columns={"check":"check"})],ignore_index=True,sort=False),SOURCE_AUDIT)
cluster_status=CLUSTER_STABILITY.status.iloc[0] if len(CLUSTER_STABILITY) else "NOT_AVAILABLE"; modes=SOURCE_AUDIT[SOURCE_AUDIT.status.eq("PASS")].groupby("block")["mode"].first().to_dict() if len(SOURCE_AUDIT) else {}; warn=[]
if cluster_status!="STABLE": warn.append(f"clusters={cluster_status}")
snapshot_used=any(v!="LIVE" for v in modes.values()); robust_val_sig=int(VALIDATION_CORR.robust_fdr_05.sum()) if len(VALIDATION_CORR) else 0; status="OK" if not warn else "OK_WITH_WARNINGS"; acceptance_level="READY_FOR_FINAL_ACCEPTANCE" if not (ACCEPTANCE_CHECKLIST.status=="FAIL").any() else "REQUIRES_REVISION"
CANONICAL_LONG=build_canonical_long(EXPECTATIONS,EXPECT_HISTORY_POINTS,ANXIETY_CURRENT,ANXIETY_HISTORY,VALIDATION); CANONICAL_LONG.to_csv(RESULTS/"FOM_CANONICAL_LONG_FIX41.csv",index=False,encoding="utf-8-sig")
try: CANONICAL_LONG.to_parquet(RESULTS/"FOM_CANONICAL_LONG_FIX41.parquet",index=False)
except Exception: pass
FIGURES=save_figures(EXPECT_ANNUAL,EXPECTATIONS,MM_GAPS,HORIZON_CONTRASTS,ANXIETY_COMBINED,ANXIETY_CURRENT,CLUSTERS,EMBEDDING,VALIDATION_CORR,CROSS_COMPONENT,CHANGEPOINTS,ML_METRICS,SHAP_IMPORTANCE); tick(4,"графики")
AUDIT={"status":status,"acceptance_level":acceptance_level,"warnings":"; ".join(warn),"expectation_group_rows":len(EXPECTATIONS),"core_2026_overall":len(EXPECTATIONS[(EXPECTATIONS.source_id.isin([15317,15337,15357,15380]))&(EXPECTATIONS.group_type.eq("overall"))]),"expectation_annual_rows":len(EXPECT_ANNUAL),"trend_rows":len(TRENDS),"changepoint_rows":len(CHANGEPOINTS),"changepoint_engine":"EXACT_DP_L2","gap_exact_permutation_tests":int(GAP_CORR.test_method.eq("EXACT_PERMUTATION").sum()),"anxiety_history_rows":len(ANXIETY_HISTORY),"anxiety_current_rows":len(ANXIETY_CURRENT),"anxiety_trend_rows":len(ANXIETY_TRENDS),"validation_rows":len(VALIDATION),"validation_tests":len(VALIDATION_CORR),"validation_fdr_sig":int(VALIDATION_CORR.significant_fdr_05.sum()) if len(VALIDATION_CORR) else 0,"validation_robust_fdr_sig":robust_val_sig,"cross_component_tests":len(CROSS_COMPONENT),"cross_component_fdr_sig":int(CROSS_COMPONENT.significant_fdr_05.sum()) if len(CROSS_COMPONENT) else 0,"cluster_groups":len(CLUSTERS),"cluster_status":cluster_status,"micro_candidates":len(MICRO_CANDIDATES),"ml_status":ML_STATUS.get("status",""),"figures":len(FIGURES),"canonical_rows":len(CANONICAL_LONG),"source_modes":json.dumps(modes,ensure_ascii=False),"snapshot_used":snapshot_used}
SHEETS={"experiment_audit":pd.DataFrame([AUDIT]),"acceptance_checklist":ACCEPTANCE_CHECKLIST,"quality_checks":QC,"source_audit":SOURCE_AUDIT,"live_audit":LIVE_AUDIT,"snapshot_audit":SNAPSHOT_AUDIT,"expectations_group":EXPECTATIONS,"expect_history_points":EXPECT_HISTORY_POINTS,"expect_annual":EXPECT_ANNUAL,"micro_macro_gaps":MM_GAPS,"horizon_contrasts":HORIZON_CONTRASTS,"gap_corr_permutation":GAP_CORR,"trends_theilsen":TRENDS,"changepoints_exact_dp":CHANGEPOINTS,"anxiety_current":ANXIETY_CURRENT,"anxiety_history":ANXIETY_HISTORY,"anxiety_combined":ANXIETY_COMBINED,"anxiety_trends":ANXIETY_TRENDS,"validation":VALIDATION,"validation_blocked":VALIDATION_CORR,"validation_by_group_type":VALIDATION_BY_BLOCK,"cross_components_perm":CROSS_COMPONENT,"clusters":CLUSTERS,"cluster_stability":CLUSTER_STABILITY,"cluster_loo":CLUSTER_LOO,"cluster_seeds":CLUSTER_SEEDS,"silhouette":SILHOUETTE,"group_matrix_raw":GROUP_MATRIX_RAW,"group_matrix_imputed":GROUP_MATRIX_IMPUTED,"embedding":EMBEDDING,"micro_registry":MICRO_REGISTRY,"ml_metrics":ML_METRICS,"ml_predictions":ML_PREDICTIONS,"feature_importance":FEATURE_IMPORTANCE,"shap_importance":SHAP_IMPORTANCE,"ml_status":pd.DataFrame([ML_STATUS])}; write_excel(RESULTS/"CHAPTER1_ANALYSIS_FIX41.xlsx",SHEETS); write_excel(RESULTS/"EXPERIMENT_AUDIT_FIX41.xlsx",{"audit":pd.DataFrame([AUDIT]),"acceptance_checklist":ACCEPTANCE_CHECKLIST,"quality_checks":QC,"source_audit":SOURCE_AUDIT,"gap_corr_permutation":GAP_CORR,"cluster_stability":CLUSTER_STABILITY,"validation_blocked":VALIDATION_CORR,"validation_by_group_type":VALIDATION_BY_BLOCK,"cross_components_perm":CROSS_COMPONENT})
for name,df in [("expectations_group",EXPECTATIONS),("expectations_annual",EXPECT_ANNUAL),("micro_macro_gaps",MM_GAPS),("horizon_contrasts",HORIZON_CONTRASTS),("gap_corr_permutation",GAP_CORR),("trends",TRENDS),("changepoints_exact_dp",CHANGEPOINTS),("anxiety_current",ANXIETY_CURRENT),("anxiety_history",ANXIETY_HISTORY),("anxiety_trends",ANXIETY_TRENDS),("validation",VALIDATION),("validation_blocked",VALIDATION_CORR),("validation_by_group_type",VALIDATION_BY_BLOCK),("cross_components_perm",CROSS_COMPONENT),("clusters",CLUSTERS),("cluster_stability",CLUSTER_STABILITY),("cluster_loo",CLUSTER_LOO),("cluster_seeds",CLUSTER_SEEDS),("embedding",EMBEDDING),("ml_metrics",ML_METRICS)]:
    if df is not None and len(df): df.to_csv(TABLES/f"{name}.csv",index=False,encoding="utf-8-sig")
(RESULTS/"METHODS_AND_RESULTS_FIX41.md").write_text(markdown_report(AUDIT,QC,ACCEPTANCE_CHECKLIST,GAP_CORR,TRENDS,CHANGEPOINTS,CLUSTER_STABILITY,VALIDATION_CORR,VALIDATION_BY_BLOCK,CROSS_COMPONENT,ML_METRICS),encoding="utf-8")
RESULT_MANIFEST=result_file_manifest(); RESULT_MANIFEST.to_csv(RESULTS/"RESULT_FILE_MANIFEST_FIX41.csv",index=False,encoding="utf-8-sig"); manifest={"created_utc":pd.Timestamp.utcnow().isoformat(),"seed":SEED,"config":CONFIG,"audit":AUDIT,"snapshot_zip_sha256":SNAPSHOT_ZIP_SHA256,"snapshot_files":SNAPSHOT_FILE_SHA256,"snapshot_origin_meta":SNAPSHOT_ORIGIN_META,"figures":FIGURES,"result_files":RESULT_MANIFEST.to_dict("records")}; (RESULTS/"MANIFEST_FIX41.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
FULL_ZIP=Path(shutil.make_archive("/content/CHAPTER1_FULL_FIX41","zip",root_dir=ROOT)); research=Path("/content/CHAPTER1_RESEARCH_FIX41"); shutil.rmtree(research,ignore_errors=True); shutil.copytree(RESULTS,research); RESEARCH_ZIP=Path(shutil.make_archive("/content/CHAPTER1_RESEARCH_FIX41","zip",root_dir=research)); PACKAGE_AUDIT=zip_audit(RESEARCH_ZIP,FULL_ZIP); PACKAGE_AUDIT.to_csv("/content/PACKAGE_AUDIT_FIX41.csv",index=False,encoding="utf-8-sig"); tick(7,"экспорт / archive audit"); progress.close()
SUMMARY=pd.DataFrame({"Показатель":["Статус","Приемочный уровень","Ядро 2026","Годовые ряды","Theil–Sen","Change-point метод","Exact permutation малых n","Тревоги 2026","Групп в кластерах","Устойчивость кластеров","Blocked validation тестов","FDR валидизация","Robust FDR валидизация","Ожидания↔тревоги FDR","ML","Графиков","Snapshot использован","Архивы"],"Значение":[AUDIT["status"],AUDIT["acceptance_level"],f'{AUDIT["core_2026_overall"]}/16',AUDIT["expectation_annual_rows"],AUDIT["trend_rows"],AUDIT["changepoint_engine"],AUDIT["gap_exact_permutation_tests"],AUDIT["anxiety_current_rows"],AUDIT["cluster_groups"],AUDIT["cluster_status"],AUDIT["validation_tests"],AUDIT["validation_fdr_sig"],AUDIT["validation_robust_fdr_sig"],AUDIT["cross_component_fdr_sig"],AUDIT["ml_status"],AUDIT["figures"],AUDIT["snapshot_used"],PACKAGE_AUDIT.status.iloc[0]]}); display(SUMMARY)

НИР — глава 1 FIX4.1:   0%|          | 0/100 [00:00<?] 

,Показатель,Значение
0,Статус,OK_WITH_WARNINGS
1,Приемочный уровень,READY_FOR_FINAL_ACCEPTANCE
2,Ядро 2026,16/16
3,Годовые ряды,40
4,Theil–Sen,4
5,Change-point метод,EXACT_DP_L2
6,Exact permutation малых n,4
7,Тревоги 2026,1750
8,Групп в кластерах,34
9,Устойчивость кластеров,MODERATE


## 4. Правила чтения финального результата

1. `acceptance_checklist` — обязательный шлюз финальной приемки; любой `FAIL` останавливает эксперимент до упаковки результатов.
2. `gap_corr_permutation` — основные связи макро/микро и горизонтов с exact/Monte-Carlo permutation p-values; асимптотический p хранится только для сопоставления.
3. `changepoints_exact_dp` — структурные сдвиги по точному penalized dynamic programming L2; результат является описательным при 10 годовых точках.
4. `validation_blocked` — основная внешняя валидизация; `robust_fdr_05` требует FDR<0.05, минимум 4 пригодных `group_type` и согласованность направления не менее 70%.
5. `validation_by_group_type` — прозрачная расшифровка того, из каких систем социального разбиения складывается внешний результат.
6. `cluster_stability` — кластеры нельзя интерпретировать без bootstrap/LOO/seed ARI; `MODERATE` означает предварительную типологию.
7. `PACKAGE_AUDIT_FIX41.csv` проверяет, что исследовательский ZIP является хэш-согласованным подмножеством `results/` полного архива.

In [ ]:
# 7. ССЫЛКИ НА РЕЗУЛЬТАТЫ FIX4.1 FINAL
from IPython.display import FileLink,display
for p in [RESULTS/"CHAPTER1_ANALYSIS_FIX41.xlsx",RESULTS/"EXPERIMENT_AUDIT_FIX41.xlsx",RESULTS/"METHODS_AND_RESULTS_FIX41.md",RESULTS/"MANIFEST_FIX41.json",Path("/content/PACKAGE_AUDIT_FIX41.csv"),Path("/content/CHAPTER1_RESEARCH_FIX41.zip"),Path("/content/CHAPTER1_FULL_FIX41.zip")]:
    if p.exists(): display(FileLink(str(p)))

/content/MONOGRAPH_CH1_FIX41/results/CHAPTER1_ANALYSIS_FIX41.xlsx

/content/MONOGRAPH_CH1_FIX41/results/EXPERIMENT_AUDIT_FIX41.xlsx

/content/MONOGRAPH_CH1_FIX41/results/METHODS_AND_RESULTS_FIX41.md

/content/MONOGRAPH_CH1_FIX41/results/MANIFEST_FIX41.json

/content/PACKAGE_AUDIT_FIX41.csv

/content/CHAPTER1_RESEARCH_FIX41.zip

/content/CHAPTER1_FULL_FIX41.zip